In [92]:
import re


def parse_logs(filepath):
    results = []
    with open(filepath, 'r', encoding='utf-8') as f:
        content = f.read()

    # 每个实验的分块，以 "RESULT" 作为分隔
    experiments = content.split("RESULT")
    print(experiments[0])
    print('--------------')
    print(experiments[1])
    

    for exp in experiments:
        # 提取参数
        params = dict(re.findall(r"args\.(\w+)\s*=\s*([^\n]+)", exp))

        # 提取 best
        best_match = re.search(r"best\s+(\d+)", exp)
        best_epoch = int(best_match.group(1)) if best_match else None

        # 提取指标
        match = re.search(
            r"ARI\s+([0-9.]+)\s+NMI\s+([0-9.]+)\s+ACC\s+([0-9.]+)\s+F1\s+([0-9.]+)", exp
        )
        if match:
            ari, nmi, acc, f1 = map(float, match.groups())
            result = {
                "params": params,
                "best": best_epoch,
                "ARI": ari,
                "NMI": nmi,
                "ACC": acc,
                "F1": f1
            }
            results.append(result)

    return results


def filter_high_ari(results, threshold=90.0):
    return [r for r in results if r["ARI"] > threshold]





In [93]:
def filter_high_bs(results, threshold=256):
    # for r in results:
    #     print(r["params"]["batch_size"])
    #     print(int(r["params"]["batch_size"])>threshold)
    return [r for r in results if int(r["params"]["batch_size"]) == threshold]

In [120]:
file_name = "Quake_Smart-seq2_Diaphragm_search_2025091218_log.txt"
file_name = 'Muraro_search_2025091218_log.txt'
file_name = 'Quake_10x_Bladder_search_2025091218_log.txt'
file_name = 'Baron3_search_2025091411_log.txt'
file_name = 'Quake_10x_Limb_Muscle_search_2025091323_log.txt'
file_name = 'Quake_10x_Bladder_search_2025091218_log.txt'
file_name = 'Quake_Smart-seq2_Limb_Muscle_search_2025091522_log.txt'
file_name = 'Chung_search_2025091523_log.txt'
file_name = 'Klein_search_2025091522_log.txt'
print(file_name)
match = re.match(r"^(.*?)_search_\d+_log\.txt$", file_name)

if match:
    dataset_name = match.group(1)
    file_name = f"{dataset_name}/{file_name}"
    print(file_name)


filepath = 'training_logs/' + file_name
all_results = parse_logs(filepath)

Klein_search_2025091522_log.txt
Klein/Klein_search_2025091522_log.txt
[2025-09-15 22:57:11] Epoch 1: Loss: 68.05489, ins: 18.46292, clu: 43.75409, pro: 5.83789
[2025-09-15 22:57:11]      acc: 0.1581, nmi: 0.1581, ari: 0.1155, f1: 0.3959
[2025-09-15 22:57:17] Epoch 20: Loss: 25.42359, ins: 16.29308, clu: 5.16048, pro: 3.97004
[2025-09-15 22:57:17]      acc: 0.7740, nmi: 0.7740, ari: 0.7593, f1: 0.6670
[2025-09-15 22:57:22] Epoch 40: Loss: 23.35875, ins: 16.12948, clu: 3.80673, pro: 3.42254
[2025-09-15 22:57:22]      acc: 0.7488, nmi: 0.7488, ari: 0.7465, f1: 0.6532
[2025-09-15 22:57:27] Epoch 60: Loss: 19.01637, ins: 15.92776, clu: 0.82584, pro: 2.26277
[2025-09-15 22:57:27]      acc: 0.7703, nmi: 0.7703, ari: 0.7788, f1: 0.7501
[2025-09-15 22:57:32] Epoch 80: Loss: 18.08116, ins: 15.78897, clu: 0.07193, pro: 2.22027
[2025-09-15 22:57:32]      acc: 0.7437, nmi: 0.7437, ari: 0.7489, f1: 0.7540
[2025-09-15 22:57:37] Epoch 100: Loss: 17.94654, ins: 15.71044, clu: 0.02753, pro: 2.20857
[202

## 统计best epoch信息

In [121]:
def best_stats(results):
    best_values = [r["best"] for r in results if r["best"] is not None]
    if not best_values:
        return None, None  # 没有数据

    max_best = max(best_values)
    avg_best = sum(best_values) / len(best_values)
    return max_best, avg_best


# 使用示例
max_best, avg_best = best_stats(all_results)
print(f"best 最大值: {max_best}")
print(f"best 平均值: {avg_best:.2f}")


best 最大值: 386
best 平均值: 214.97


## 筛选ARI分数

In [127]:
ari_threshold = 83

ari_results = filter_high_ari(all_results, threshold=ari_threshold)


print_results = ari_results

if not print_results:
    print(f"没有找到 ARI > {ari_threshold} 的实验。")
else:
    for i, r in enumerate(print_results, 1):
        print(f"\n实验 {i}:")
        print("参数:")
        for k, v in r["params"].items():
            print(f"args.{k} = {v}")
            if "batch_size" in k:
                print()
        print("============结果============:")
        print(f"best    {r['best']}")
        print(f"ARI    {r['ARI']}")
        print(f"NMI    {r['NMI']}")
        print(f"ACC    {r['ACC']}")
        print(f"F1     {r['F1']}")


实验 1:
参数:
args.temperature = 0.4
args.k = 0.8
args.n_neighbors = 4
args.batch_size = 512

args.dropout = 0.9
args.lambda_c = 1.0
args.lambda_p = 1.0
args.lr = 0.003
args.seed = 100
args.noise = 0.0
args.m = 0.5
============结果============:
best    26
ARI    83.03
NMI    78.13
ACC    90.76
F1     87.51

实验 2:
参数:
args.temperature = 0.2
args.k = 0.4
args.n_neighbors = 7
args.batch_size = 256

args.dropout = 0.9
args.lambda_c = 1.0
args.lambda_p = 1.0
args.lr = 0.003
args.seed = 100
args.noise = 0.0
args.m = 0.5
============结果============:
best    344
ARI    83.23
NMI    79.18
ACC    92.12
F1     90.12

实验 3:
参数:
args.temperature = 0.2
args.k = 0.4
args.n_neighbors = 7
args.batch_size = 256

args.dropout = 0.9
args.lambda_c = 1.0
args.lambda_p = 1.0
args.lr = 0.003
args.seed = 100
args.noise = 0.0
args.m = 0.5
============结果============:
best    344
ARI    83.23
NMI    79.18
ACC    92.12
F1     90.12

实验 4:
参数:
args.temperature = 0.2
args.k = 0.4
args.n_neighbors = 7
args.batch_size = 256

## 筛选batchsize

In [125]:
bs_threshold = 512
bs_results = filter_high_bs(results, threshold=bs_threshold)

print_results = bs_results

if not print_results:
    print(f"没有找到 ARI > {ari_threshold} 的实验。")

    print(f"没有找到 batchsize == {bs_threshold} 的实验。")
else:
    for i, r in enumerate(print_results, 1):
        print(f"\n实验 {i}:")
        print("参数:")
        for k, v in r["params"].items():
            print(f"args.{k} = {v}")
            if "batch_size" in k:
                print()
        print("============结果============:")
        print(f"best    {r['best']}")
        print(f"ARI    {r['ARI']}")
        print(f"NMI    {r['NMI']}")
        print(f"ACC    {r['ACC']}")
        print(f"F1     {r['F1']}")

没有找到 ARI > 73 的实验。
没有找到 batchsize == 512 的实验。
